
# ILLM Lab 6 — Language Identification: 5 Classifiers Compared by Macro-F1

Trains and compares five language-identification classifiers on the language ID
dataset from Assignment 1, using pretrained FastText Wikipedia word vectors as
features:

1. **Logistic Regression** on averaged word vectors
2. **Multi-Layer Perceptron**, with 0, 1, and 2 hidden layers
3. **LSTM**
4. **GRU**
5. **FastText supervised classification** (the `fasttext` library's own trainer)

Every evaluation metric (precision, recall, macro-F1) is implemented **from
scratch** — no `sklearn.metrics` — per the assignment's explicit constraint.

---

## 1. Deep analysis of the assignment

| # | Requirement | What it means concretely |
|---|---|---|
| 1 | Use the language identification dataset from Assignment 1 | A labeled dataset of (text, language) pairs, covering multiple languages — see the **important note** right below this table. |
| 2 | Macro-F1 evaluation, **no predefined sklearn classes for evaluation** | Precision, recall, and F1 must be computed per class from raw counts, then macro-averaged (unweighted mean across classes) by hand — implemented once in Section 4 and reused for every classifier. |
| 3 | Use FastText wiki vectors for different languages | Download per-language `.vec` files from the [fastText pretrained vectors page](https://fasttext.cc/docs/en/pretrained-vectors.html) and use them as fixed (frozen), pretrained word embeddings — not vectors trained from scratch on the assignment's own data. |
| 4.1 | Logistic Regression, averaged word vectors as features | Mean of a sentence's word vectors → a single fixed-size vector → linear classifier. |
| 4.2 | MLP, $n=0,1,2$ hidden layers | Same averaged-vector input as #1, through a feedforward network with $n$ hidden layers. **$n=0$ hidden layers *is* logistic regression** — same input, same linear-then-softmax structure — so this notebook implements one configurable feedforward classifier and reuses it for both requirements, rather than writing the same architecture twice (see Section 6's design-decision note). |
| 4.3 / 4.4 | LSTM / GRU | These need the *sequence* of per-word vectors (order-sensitive), not a single averaged vector — a fundamentally different input pipeline from #1–2, requiring padding and batching. |
| 4.5 | FastText supervised classification | Uses the `fasttext` library's own `train_supervised`, following the [official tutorial](https://fasttext.cc/docs/en/supervised-tutorial.html) — a self-contained classifier, initialized with the same pretrained wiki vectors via fastText's `pretrainedVectors` option so requirement #3 still applies here too. |
| 5 | Compare all classifiers, report macro-F1 | One shared evaluation function (Section 4), applied identically to every classifier's predictions, assembled into one final table (Section 11). |

### ⚠️ Important note on the dataset

Unlike Labs 3–5 — which all reused the *same*, already-known 1,000,000-sentence
single-language corpus from Assignment 1 — **language identification inherently
needs multiple labeled languages**, and this conversation has no prior
information about what your Assignment-1 language-ID dataset actually looks
like (its file format, which languages, how large). Rather than guess wrong and
build something that silently doesn't match your real data, **Section 2 below
is a configurable loader**:

- It first looks for `langid_train.csv` / `langid_val.csv` / `langid_test.csv`
  (each with `text` and `label` columns) in the working directory — **point
  this at your real Assignment-1 files by saving/renaming them to these three
  filenames** (or edit the paths at the top of Section 2).
- If those files aren't found, it falls back to a small **built-in demo
  dataset** (a handful of languages, real short sentences) purely so that
  every cell in this notebook is runnable and demonstrably correct end-to-end
  without your file. **The demo dataset is not a substitute for your real
  data** — swap in your actual files before trusting the reported macro-F1
  scores for anything beyond "does the pipeline run."

**Design decisions (stated explicitly, since the assignment leaves these open):**

- **Logistic Regression and MLP ($n=0,1,2$) are all implemented in PyTorch**,
  not `sklearn.linear_model.LogisticRegression` / `sklearn.neural_network.MLPClassifier`.
  The evaluation-only sklearn restriction wouldn't technically forbid using
  sklearn to *train* these two, but LSTM and GRU have no sklearn equivalent at
  all — implementing all four in the same framework, with the same optimizer
  and training loop, makes the four-way comparison architecturally fair
  (same training regime, not "two different libraries' idea of a good default").
- **FastText `.vec` files are streamed and truncated to the most frequent
  `MAX_VOCAB_PER_LANG` words per language** (default 100,000), not downloaded
  in full. These files are frequency-sorted and can be multiple gigabytes each
  (English alone is several GB) — streaming the HTTP response and stopping
  once enough lines are read, rather than downloading the whole file, is the
  standard practical mitigation and keeps this runnable on a laptop.
- **A word absent from every downloaded vector file is skipped** when
  averaging (Logistic Regression / MLP / LSTM / GRU input) — there is no
  `<UNK>` vector to fall back to, since these are frozen pretrained embeddings,
  not something trained on this dataset.
- **Two environment-specific fixes**, found by actually running this pipeline
  during development, are included in Section 1 and called out there:
  `fasttext` 0.9.3 crashes under NumPy ≥ 2.0 (installing `fasttext-wheel` and
  pinning `numpy<2` fixes it), and `fasttext.train_supervised` can crash with
  a low-level floating-point exception on machines where its default thread
  count doesn't match the actual CPU count (fixed by passing
  `thread=os.cpu_count()` explicitly). Both are real, reproducible issues, not
  hypothetical ones — worth knowing about if you hit them too.

## 2. Plan for this notebook

```
Load the language-ID dataset (real files if present, else a small demo set)
        ▼
Download + truncate FastText wiki vectors for each language in the dataset
        ▼
Implement macro-F1 from scratch (precision/recall/F1 per class, unweighted mean)
        ▼
Build two input representations from the same word vectors:
  (a) averaged sentence vectors      -- for Logistic Regression & MLP
  (b) padded per-token sequences     -- for LSTM & GRU
        ▼
Train & evaluate: Logistic Regression / MLP(n=0,1,2) / LSTM / GRU / FastText-supervised
        ▼
Compare all classifiers' macro-F1 in one table, on validation and test
```



## Section 1 — Environment Setup

Installs the exact package combination that avoids the two `fasttext` bugs
described above.


In [ ]:

# fasttext 0.9.3 (the package on PyPI as `fasttext`) crashes under NumPy >= 2.0.
# fasttext-wheel is a maintained rebuild that fixes the training-side crash;
# NumPy is pinned below 2.0 to also fix a predict()-side incompatibility.
!pip install --quiet fasttext-wheel "numpy<2" torch pandas scikit-learn datasets


In [ ]:

import os
import re
import time
import random
import urllib.request
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
import fasttext

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# The number of CPU threads actually available -- passed explicitly to every
# fasttext.train_supervised call below. fastText's own default thread count
# doesn't check the real CPU count, and can crash with a low-level floating-
# point exception (not even a catchable Python exception) if it oversubscribes
# a machine with few cores. This one line is the fix.
N_THREADS = os.cpu_count() or 1

VECTOR_DIM = 300           # fastText wiki vectors are published at 300 dimensions
MAX_VOCAB_PER_LANG = 100_000  # truncate each downloaded .vec file to its N most frequent words
HIDDEN_DIM = 64             # hidden layer width for MLP / LSTM / GRU

print(f"Using {N_THREADS} thread(s). NumPy {np.__version__}, PyTorch {torch.__version__}.")



## Section 2 — Build the Language-ID Dataset from IndicCorpV2

**Correction from an earlier version of this notebook:** language
identification inherently needs more than one language, but every file
you've used through Labs 3–5 (`hin_Deva_sentences.csv`,
`hin_Deva_tokenizer_results.pkl`) is Hindi-only — there is no pre-existing
multi-language dataset from "Assignment 1" to reload. Per your confirmation,
this section instead **builds one**, by extending Lab 3's own data source —
[`ai4bharat/IndicCorpV2`](https://huggingface.co/datasets/ai4bharat/IndicCorpV2)
on HuggingFace — to a handful of additional languages alongside Hindi, using
the same `<lang>_<script>` config naming Lab 3 already used (`hin_Deva`).

**Design decisions (stated explicitly):**

- **Languages chosen:** `hin_Deva` (Hindi), `mar_Deva` (Marathi), `ben_Beng`
  (Bengali), `tam_Taml` (Tamil), `urd_Arab` (Urdu) — a deliberately mixed set,
  not five arbitrary ones. Hindi and Marathi **share the same script**
  (Devanagari), so distinguishing them forces a classifier to actually learn
  lexical/subword differences rather than just recognizing a character set —
  a meaningfully *harder* pair than the rest. Bengali and Tamil use entirely
  distinct scripts (easy on script alone). Urdu is written in a modified
  Arabic script but is lexically/grammatically very close to Hindi
  (colloquial Hindi and Urdu are largely mutually intelligible when spoken) —
  another interesting contrast. Edit `LANGUAGES_TO_COLLECT` below to use a
  different set.
- **`SENTENCES_PER_LANG = 10,000`** (50,000 total) — enough to train all five
  classifier families without needing anywhere near Lab 3's full
  1,000,000-sentence scale, which isn't necessary for a classification task
  with only 5 classes. Change it if you want more.
- **Sentence splitting:** IndicCorpV2's `text` format is raw
  paragraph/document-level text, not pre-split into sentences, so a simple
  regex splits on `।` (Devanagari/Bengali danda), `۔` (Arabic/Urdu full
  stop), and `.`/`!`/`?`, then keeps only results with 3–50 words — the same
  "reasonable sentence length" filter used for the length-stratified split in
  Lab 3.
- **Streamed, not fully downloaded**, exactly like Lab 3: `datasets.load_dataset(..., streaming=True)`
  reads records one at a time and collection stops as soon as
  `SENTENCES_PER_LANG` valid, deduplicated sentences have been found for that
  language — no need to pull an entire language's corpus to disk first.
- **Split:** 80% train / 10% validation / 10% test, stratified by language
  (so every split has the same per-language balance) — a simple, standard
  split for a classification task, unlike Lab 3's length-stratified dev/test
  design (which existed for a different reason: keeping tokenizer evaluation
  representative of the true sentence-length distribution).

> **Expect the first run of this cell to take a while** — it's a real,
> multi-language network download. Results are checkpointed to
> `langid_train.csv` / `langid_val.csv` / `langid_test.csv`, so re-running is
> instant afterward, and — per Section 1's original design — if you already
> have your own separate labeled language-ID file, just save it under those
> three filenames and this cell will pick it up unchanged instead of
> re-collecting from IndicCorpV2.


In [ ]:

TRAIN_PATH = "langid_train.csv"
VAL_PATH = "langid_val.csv"
TEST_PATH = "langid_test.csv"

LANGUAGES_TO_COLLECT = {
    "hin_Deva": "Hindi",
    "mar_Deva": "Marathi",
    "ben_Beng": "Bengali",
    "tam_Taml": "Tamil",
    "urd_Arab": "Urdu",
}
SENTENCES_PER_LANG = 10_000
MIN_WORDS, MAX_WORDS = 3, 50

SENT_SPLIT_RE = re.compile(r"[।۔.!?]+")

def split_into_sentences(text):
    parts = SENT_SPLIT_RE.split(str(text))
    return [p.strip() for p in parts if p.strip()]

def collect_language(lang_code, target_count):
    """Stream ai4bharat/IndicCorpV2 for one language config, splitting each
    record into sentences, deduplicating, and stopping as soon as enough
    valid sentences are found -- never downloads the full split to disk."""
    from datasets import load_dataset
    seen = set()
    collected = []
    stream = load_dataset("ai4bharat/IndicCorpV2", "indiccorp_v2",
                           data_dir=f"data/{lang_code}", split=lang_code, streaming=True)
    for example in stream:
        for sent in split_into_sentences(example["text"]):
            n_words = len(sent.split())
            if n_words < MIN_WORDS or n_words > MAX_WORDS:
                continue
            if sent in seen:
                continue
            seen.add(sent)
            collected.append(sent)
            if len(collected) >= target_count:
                return collected
    return collected  # fewer than target_count if the language's data runs out first

if all(os.path.exists(p) for p in (TRAIN_PATH, VAL_PATH, TEST_PATH)):
    train_df = pd.read_csv(TRAIN_PATH)
    val_df = pd.read_csv(VAL_PATH)
    test_df = pd.read_csv(TEST_PATH)
    print(f"Loaded cached dataset from {TRAIN_PATH} / {VAL_PATH} / {TEST_PATH}.")
else:
    print("Collecting a fresh multi-language dataset from ai4bharat/IndicCorpV2...")
    all_rows = []
    for lang_code, lang_name in LANGUAGES_TO_COLLECT.items():
        t0 = time.time()
        sentences = collect_language(lang_code, SENTENCES_PER_LANG)
        print(f"  {lang_code:<10} ({lang_name:<8}): {len(sentences):,} sentences "
              f"in {time.time()-t0:.1f}s  e.g. {sentences[0][:40]!r}")
        all_rows.extend((s, lang_code) for s in sentences)

    full_df = pd.DataFrame(all_rows, columns=["text", "label"])

    # stratified 80/10/10 split: shuffle within each language, then slice
    train_parts, val_parts, test_parts = [], [], []
    rng = random.Random(RANDOM_SEED)
    for lang_code in LANGUAGES_TO_COLLECT:
        rows = full_df[full_df["label"] == lang_code].sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
        n = len(rows)
        n_train = int(n * 0.8)
        n_val = int(n * 0.9)
        train_parts.append(rows.iloc[:n_train])
        val_parts.append(rows.iloc[n_train:n_val])
        test_parts.append(rows.iloc[n_val:])

    train_df = pd.concat(train_parts).sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
    val_df = pd.concat(val_parts).sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
    test_df = pd.concat(test_parts).sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

    train_df.to_csv(TRAIN_PATH, index=False)
    val_df.to_csv(VAL_PATH, index=False)
    test_df.to_csv(TEST_PATH, index=False)
    print(f"\\nSaved {TRAIN_PATH} / {VAL_PATH} / {TEST_PATH}.")

LANGUAGES = sorted(train_df["label"].unique().tolist())
print(f"\\nLanguages: {LANGUAGES}")
print(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")
train_df.head()



## Section 3 — Download FastText Wiki Vectors

One `.vec` file per language actually present in the dataset (`LANGUAGES`,
from Section 2), from the [official pretrained-vectors page](https://fasttext.cc/docs/en/pretrained-vectors.html):
`https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.<code>.vec`.

**Streamed and truncated, not fully downloaded.** These files list every word
in **frequency order** (documented on the page above), so keeping only the
first `MAX_VOCAB_PER_LANG` lines keeps the most useful (most frequent) words
while avoiding downloading multiple gigabytes per language. The HTTP
connection is closed as soon as enough lines have been read.

> **Language code note:** fastText's codes are (mostly) standard two-letter
> ISO codes. If `LANGUAGES` from Section 2 uses different codes than fastText
> does for one of your languages, edit `LANG_CODE_MAP` below.

> **Expect this cell to take a while the first time** — even truncated,
> `MAX_VOCAB_PER_LANG=100,000` at 300 dimensions is tens of megabytes *per
> language*, and it's a real network download. Every file is cached to disk
> afterward, so re-running the cell is instant.


In [ ]:

LANG_CODE_MAP = {
    # IndicCorpV2 config name -> fastText pretrained-vector language code.
    # Only relevant if LANGUAGES (Section 2) uses IndicCorpV2-style codes;
    # edit/extend this if you used a different language set or a dataset
    # with its own different code scheme.
    "hin_Deva": "hi",
    "mar_Deva": "mr",
    "ben_Beng": "bn",
    "tam_Taml": "ta",
    "urd_Arab": "ur",
}

def fasttext_wiki_url(lang_code):
    return f"https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.{lang_code}.vec"

def download_and_truncate_vec(url, out_path, max_words):
    """Stream a fastText .vec file and keep only the first max_words lines
    (already frequency-sorted), instead of downloading the full file."""
    if os.path.exists(out_path):
        return out_path
    print(f"  Downloading (streaming, truncating to {max_words:,} words): {url}")
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp:
        header = resp.readline().decode("utf-8")
        total_words, dim = map(int, header.split())
        kept = []
        for i, raw_line in enumerate(resp):
            if i >= max_words:
                break
            kept.append(raw_line.decode("utf-8", errors="ignore").rstrip("\n"))
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(f"{len(kept)} {dim}\n")
        for line in kept:
            f.write(line + "\n")
    print(f"    -> saved {len(kept):,} words to {out_path}")
    return out_path

def load_vec_file(path):
    vecs = {}
    with open(path, encoding="utf-8") as f:
        f.readline()  # header
        for line in f:
            parts = line.rstrip("\n").split(" ")
            word = parts[0]
            try:
                vecs[word] = np.array(parts[1:], dtype=np.float32)
            except ValueError:
                continue  # skip any malformed line rather than crash the whole load
    return vecs

os.makedirs("fasttext_vecs", exist_ok=True)
word_vectors = {}          # combined lookup across every language
vec_file_paths = {}        # per-language path, needed later to build a merged file

for lang in LANGUAGES:
    ft_code = LANG_CODE_MAP.get(lang, lang)
    out_path = f"fasttext_vecs/wiki.{ft_code}.vec"
    download_and_truncate_vec(fasttext_wiki_url(ft_code), out_path, MAX_VOCAB_PER_LANG)
    vec_file_paths[lang] = out_path
    lang_vecs = load_vec_file(out_path)
    word_vectors.update(lang_vecs)
    print(f"  {lang}: {len(lang_vecs):,} word vectors loaded")

print(f"\nCombined vocabulary across all languages: {len(word_vectors):,} words")



## Section 4 — Macro-F1 From Scratch

Per the assignment: **no `sklearn.metrics`**. For each class $c$, from raw
counts against the true labels:

$$\text{precision}_c = \frac{TP_c}{TP_c + FP_c} \qquad
\text{recall}_c = \frac{TP_c}{TP_c + FN_c} \qquad
F1_c = \frac{2 \cdot \text{precision}_c \cdot \text{recall}_c}{\text{precision}_c + \text{recall}_c}$$

(all three defined as $0$ if their denominator is $0$ — a class with no
predicted instances has undefined precision, not infinite or crashing
precision). **Macro-F1** is simply the unweighted mean of $F1_c$ across every
class — every language counts equally regardless of how many test examples
it has, which is exactly what makes "macro" different from "micro" or
"weighted" averaging.

$$\text{macro-F1} = \frac{1}{|C|}\sum_{c \in C} F1_c$$


In [ ]:

def evaluate_predictions(y_true, y_pred, labels):
    """From-scratch precision/recall/F1 per class + macro-F1. No sklearn.metrics."""
    per_class = {}
    for label in labels:
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == label and p == label)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t != label and p == label)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == label and p != label)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
        per_class[label] = {"precision": precision, "recall": recall, "f1": f1, "support": tp + fn}
    macro_f1 = sum(per_class[l]["f1"] for l in labels) / len(labels)
    macro_precision = sum(per_class[l]["precision"] for l in labels) / len(labels)
    macro_recall = sum(per_class[l]["recall"] for l in labels) / len(labels)
    return {"macro_f1": macro_f1, "macro_precision": macro_precision,
            "macro_recall": macro_recall, "per_class": per_class}


In [ ]:

# Hand-checkable sanity example: 2 classes, one obviously-imperfect confusion pattern.
_y_true = ["en", "en", "en", "hi", "hi", "hi"]
_y_pred = ["en", "en", "hi", "hi", "hi", "en"]
# en: TP=2 (first two), FP=1 (the "hi" example predicted "en"), FN=1 (one "en" predicted "hi")
#     precision=2/3=0.667  recall=2/3=0.667  F1=0.667
# hi: TP=2, FP=1, FN=1 -> by symmetry, same numbers: precision=recall=F1=0.667
# macro-F1 = mean(0.667, 0.667) = 0.667
_result = evaluate_predictions(_y_true, _y_pred, ["en", "hi"])
print("Sanity check (hand-computed expected macro-F1 ~= 0.667):")
print("  computed macro-F1 = " + format(_result["macro_f1"], ".4f"))
assert abs(_result["macro_f1"] - 2/3) < 1e-9, "Macro-F1 implementation disagrees with the hand-worked example!"
print("  matches hand-worked example.")



## Section 5 — Preprocessing: Two Input Representations From the Same Vectors

Every classifier reads from the same `word_vectors` lookup built in Section 3,
but Logistic Regression/MLP need a **single averaged vector per sentence**,
while LSTM/GRU need the **full padded sequence of per-token vectors** — these
are genuinely different tensors, built once here and reused by every model
that needs that shape.


In [ ]:

def tokenize(text):
    return str(text).lower().split()

label_to_idx = {lang: i for i, lang in enumerate(LANGUAGES)}
idx_to_label = {i: lang for lang, i in label_to_idx.items()}

def average_vector(tokens, word_vectors, dim):
    vecs = [word_vectors[w] for w in tokens if w in word_vectors]
    if not vecs:
        return np.zeros(dim, dtype=np.float32)
    return np.mean(vecs, axis=0).astype(np.float32)

def build_avg_dataset(df):
    tokens_list = [tokenize(t) for t in df["text"]]
    X = np.array([average_vector(toks, word_vectors, VECTOR_DIM) for toks in tokens_list], dtype=np.float32)
    y = np.array([label_to_idx[l] for l in df["label"]], dtype=np.int64)
    return torch.tensor(X), torch.tensor(y)

X_train_avg, y_train = build_avg_dataset(train_df)
X_val_avg, y_val = build_avg_dataset(val_df)
X_test_avg, y_test = build_avg_dataset(test_df)
print(f"Averaged-vector features: train={tuple(X_train_avg.shape)} val={tuple(X_val_avg.shape)} test={tuple(X_test_avg.shape)}")


In [ ]:

def sentence_to_seq_tensor(tokens, word_vectors, dim):
    vecs = [word_vectors[w] for w in tokens if w in word_vectors]
    if not vecs:
        vecs = [np.zeros(dim, dtype=np.float32)]  # keep every sentence non-empty for the RNN
    return torch.tensor(np.array(vecs, dtype=np.float32))

def build_seq_dataset(df):
    tokens_list = [tokenize(t) for t in df["text"]]
    seqs = [sentence_to_seq_tensor(toks, word_vectors, VECTOR_DIM) for toks in tokens_list]
    labels = torch.tensor([label_to_idx[l] for l in df["label"]], dtype=torch.long)
    lengths = torch.tensor([len(s) for s in seqs], dtype=torch.long)
    return seqs, lengths, labels

train_seqs, train_lengths, train_seq_labels = build_seq_dataset(train_df)
val_seqs, val_lengths, val_seq_labels = build_seq_dataset(val_df)
test_seqs, test_lengths, test_seq_labels = build_seq_dataset(test_df)
print(f"Sequence features: train={len(train_seqs)} val={len(val_seqs)} test={len(test_seqs)} sentences")

def collate_batch(seqs, lengths, labels, indices):
    batch_seqs = [seqs[i] for i in indices]
    batch_lengths = lengths[indices]
    batch_labels = labels[indices]
    padded = pad_sequence(batch_seqs, batch_first=True)  # (B, T_max, VECTOR_DIM)
    return padded, batch_lengths, batch_labels



## Section 6 — Tasks 1 & 2: Logistic Regression and MLP ($n=0,1,2$)

One `FeedforwardClassifier(input_dim, hidden_dims, output_dim)`, used for all
four required configurations:

| Requirement | `hidden_dims` |
|---|---|
| Logistic Regression | `[]` |
| MLP, $n=0$ | `[]` — **identical architecture to Logistic Regression** |
| MLP, $n=1$ | `[HIDDEN_DIM]` |
| MLP, $n=2$ | `[HIDDEN_DIM, HIDDEN_DIM]` |

With `hidden_dims=[]`, the network is a single `Linear(input_dim, output_dim)`
layer followed by softmax (via `CrossEntropyLoss`, which applies it
internally) — exactly logistic regression's decision rule, generalized from
binary to multi-class. This is why "Logistic Regression" and "MLP, $n=0$" are
trained **once** below and reported under both names, rather than duplicated —
they are the same model by construction, not by coincidence.


In [ ]:

class FeedforwardClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            prev = h
        layers.append(nn.Linear(prev, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def train_feedforward(hidden_dims, X_train, y_train, epochs=100, lr=0.01, weight_decay=1e-4, seed=RANDOM_SEED):
    torch.manual_seed(seed)
    model = FeedforwardClassifier(X_train.shape[1], hidden_dims, len(LANGUAGES))
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        opt.zero_grad()
        out = model(X_train)
        loss = loss_fn(out, y_train)
        loss.backward()
        opt.step()
    return model

def predict_feedforward(model, X):
    model.eval()
    with torch.no_grad():
        return model(X).argmax(dim=1).numpy()


In [ ]:

feedforward_configs = {
    "Logistic Regression": [],
    "MLP (n=0 hidden layers)": [],   # same architecture as Logistic Regression, trained once, reused below
    "MLP (n=1 hidden layer)": [HIDDEN_DIM],
    "MLP (n=2 hidden layers)": [HIDDEN_DIM, HIDDEN_DIM],
}

feedforward_models = {}
_trained_cache = {}  # keyed by the tuple(hidden_dims) so the n=0 case is only trained once

for name, hidden_dims in feedforward_configs.items():
    key = tuple(hidden_dims)
    if key not in _trained_cache:
        t0 = time.time()
        model = train_feedforward(hidden_dims, X_train_avg, y_train)
        print(f"Trained architecture hidden_dims={hidden_dims} in {time.time()-t0:.1f}s")
        _trained_cache[key] = model
    feedforward_models[name] = _trained_cache[key]

print(f"\n{len(_trained_cache)} distinct architecture(s) trained, reused across {len(feedforward_configs)} named configurations.")



## Section 7 — Tasks 3 & 4: LSTM and GRU

Both read the padded per-token sequences built in Section 5 (not the averaged
vectors) — the whole point of a recurrent model is to use word *order*, which
averaging destroys. Sequences are packed with
`pack_padded_sequence`/`pad_sequence` so the padding positions themselves
don't influence the learned weights, and the final hidden state of the
recurrent layer is fed into a linear classifier head — the same "read the
sequence, then classify from what's left at the end" architecture, with the
only difference between the two being the recurrent cell itself.


In [ ]:

class RNNClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, cell="lstm"):
        super().__init__()
        rnn_cls = nn.LSTM if cell == "lstm" else nn.GRU
        self.rnn = rnn_cls(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, hidden = self.rnn(packed)
        h_n = hidden[0] if isinstance(hidden, tuple) else hidden  # LSTM: (h_n, c_n); GRU: h_n
        return self.fc(h_n[-1])


def train_rnn(cell, epochs=30, batch_size=32, lr=0.01, seed=RANDOM_SEED):
    torch.manual_seed(seed)
    model = RNNClassifier(VECTOR_DIM, HIDDEN_DIM, len(LANGUAGES), cell=cell)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    n = len(train_seqs)
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(n)
        for start in range(0, n, batch_size):
            idx = perm[start:start + batch_size]
            xb, lb, yb = collate_batch(train_seqs, train_lengths, train_seq_labels, idx)
            opt.zero_grad()
            loss = loss_fn(model(xb, lb), yb)
            loss.backward()
            opt.step()
    return model

def predict_rnn(model, seqs, lengths):
    model.eval()
    with torch.no_grad():
        xb, lb, _ = collate_batch(seqs, lengths, torch.zeros(len(seqs), dtype=torch.long), torch.arange(len(seqs)))
        return model(xb, lb).argmax(dim=1).numpy()


In [ ]:

rnn_models = {}
for cell in ["lstm", "gru"]:
    t0 = time.time()
    rnn_models[cell] = train_rnn(cell)
    print(f"{cell.upper()} trained in {time.time()-t0:.1f}s")



## Section 8 — Task 5: FastText Supervised Classification

Following the [official supervised-classification tutorial](https://fasttext.cc/docs/en/supervised-tutorial.html):
`fasttext.train_supervised` expects a plain text file, one example per line,
formatted as `__label__<language> <text>`.

**Honoring "use FastText wiki vectors" for this classifier too:** by default,
`train_supervised` learns its own word vectors from scratch. Passing
`pretrainedVectors=` (pointed at a `.vec` file) initializes training from
existing vectors instead — the same mechanism used for tasks 1–4, applied via
fastText's own documented option. Since `pretrainedVectors` only accepts a
*single* file, the per-language files downloaded in Section 3 are concatenated
into one combined file first.

**`thread=N_THREADS` is passed explicitly on every call below** — see the
Section 1 note on why leaving it at fastText's internal default can crash
training outright on some machines.


In [ ]:

def write_fasttext_format(df, path):
    with open(path, "w", encoding="utf-8") as f:
        for text, label in zip(df["text"], df["label"]):
            clean_text = str(text).replace("\n", " ")
            f.write(f"__label__{label} {clean_text}\n")

write_fasttext_format(train_df, "ft_train.txt")
write_fasttext_format(val_df, "ft_val.txt")
write_fasttext_format(test_df, "ft_test.txt")
print("Wrote fastText-format train/val/test files.")


In [ ]:

# Merge the per-language .vec files (Section 3) into one combined file, since
# pretrainedVectors accepts only a single path.
combined_vec_path = "fasttext_vecs/combined.vec"
if not os.path.exists(combined_vec_path):
    all_lines = []
    dim = None
    for lang in LANGUAGES:
        with open(vec_file_paths[lang], encoding="utf-8") as f:
            header = f.readline()
            _, dim = map(int, header.split())
            all_lines.extend(line.rstrip("\n") for line in f)
    with open(combined_vec_path, "w", encoding="utf-8") as f:
        f.write(f"{len(all_lines)} {dim}\n")
        for line in all_lines:
            f.write(line + "\n")
    print(f"Wrote combined vector file: {len(all_lines):,} words, dim={dim}")
else:
    print(f"Combined vector file already exists: {combined_vec_path}")


In [ ]:

t0 = time.time()
fasttext_model = fasttext.train_supervised(
    input="ft_train.txt",
    pretrainedVectors=combined_vec_path,
    dim=VECTOR_DIM,
    epoch=25,
    lr=0.5,
    wordNgrams=2,
    thread=N_THREADS,     # see Section 1: fixes a low-level crash on some machines
    verbose=0,
)
print(f"FastText supervised model trained in {time.time()-t0:.1f}s")

def predict_fasttext(model, df):
    preds = []
    for text in df["text"]:
        clean_text = str(text).replace("\n", " ")
        label = model.predict(clean_text)[0][0].replace("__label__", "")
        preds.append(label)
    return preds



## Section 9 — Compare All Classifiers by Macro-F1

Every model's predictions are run through the *same* `evaluate_predictions`
function from Section 4 — nothing here uses `sklearn.metrics`.


In [ ]:

idx_to_label_arr = np.array([idx_to_label[i] for i in range(len(LANGUAGES))])

def y_true_labels(y_idx_tensor):
    return [idx_to_label[i] for i in y_idx_tensor.numpy()]

results_rows = []

# --- Logistic Regression / MLP(n=0,1,2) ---
for name, model in feedforward_models.items():
    val_pred_idx = predict_feedforward(model, X_val_avg)
    test_pred_idx = predict_feedforward(model, X_test_avg)
    val_pred = [idx_to_label[i] for i in val_pred_idx]
    test_pred = [idx_to_label[i] for i in test_pred_idx]
    val_result = evaluate_predictions(y_true_labels(y_val), val_pred, LANGUAGES)
    test_result = evaluate_predictions(y_true_labels(y_test), test_pred, LANGUAGES)
    results_rows.append({"Classifier": name, "Val Macro-F1": val_result["macro_f1"], "Test Macro-F1": test_result["macro_f1"]})

# --- LSTM / GRU ---
for cell_name, model in rnn_models.items():
    val_pred_idx = predict_rnn(model, val_seqs, val_lengths)
    test_pred_idx = predict_rnn(model, test_seqs, test_lengths)
    val_pred = [idx_to_label[i] for i in val_pred_idx]
    test_pred = [idx_to_label[i] for i in test_pred_idx]
    val_result = evaluate_predictions(y_true_labels(val_seq_labels), val_pred, LANGUAGES)
    test_result = evaluate_predictions(y_true_labels(test_seq_labels), test_pred, LANGUAGES)
    results_rows.append({"Classifier": cell_name.upper(), "Val Macro-F1": val_result["macro_f1"], "Test Macro-F1": test_result["macro_f1"]})

# --- FastText supervised ---
ft_val_pred = predict_fasttext(fasttext_model, val_df)
ft_test_pred = predict_fasttext(fasttext_model, test_df)
ft_val_result = evaluate_predictions(val_df["label"].tolist(), ft_val_pred, LANGUAGES)
ft_test_result = evaluate_predictions(test_df["label"].tolist(), ft_test_pred, LANGUAGES)
results_rows.append({"Classifier": "FastText Supervised", "Val Macro-F1": ft_val_result["macro_f1"], "Test Macro-F1": ft_test_result["macro_f1"]})

results_df = pd.DataFrame(results_rows).sort_values("Test Macro-F1", ascending=False).reset_index(drop=True)
results_df.to_csv("lab6_results.csv", index=False)   # saved so the BERT notebook (Lab 10) can compare against these models
results_df

In [ ]:

# Per-class breakdown on the TEST set for the best-performing classifier, for a closer look
best_name = results_df.iloc[0]["Classifier"]
print(f"Per-language test-set breakdown for the best classifier ({best_name}):\n")

if best_name == "FastText Supervised":
    best_result = ft_test_result
elif best_name in ("LSTM", "GRU"):
    best_model = rnn_models[best_name.lower()]
    best_pred = [idx_to_label[i] for i in predict_rnn(best_model, test_seqs, test_lengths)]
    best_result = evaluate_predictions(y_true_labels(test_seq_labels), best_pred, LANGUAGES)
else:
    best_model = feedforward_models[best_name]
    best_pred = [idx_to_label[i] for i in predict_feedforward(best_model, X_test_avg)]
    best_result = evaluate_predictions(y_true_labels(y_test), best_pred, LANGUAGES)

per_class_df = pd.DataFrame(best_result["per_class"]).T
per_class_df



## Section 10 — Discussion

Guidance for what to look for and write up, grounded in what each classifier
is built to do — fill in with the *actual* numbers from your run (and be sure
you're looking at results from your real dataset, not the Section 2 demo
fallback, before drawing conclusions):

- **Logistic Regression and MLP ($n=0$) should score identically** (Section 6)
  — if they don't in your results table, something about how they were
  trained diverged, and that's worth double-checking rather than reporting as
  a real difference.
- **Does more hidden-layer depth (n=0 → 1 → 2) actually help?** More
  capacity only helps if there's enough training data to use it — on a small
  language-ID dataset, $n=2$ overfitting and scoring *worse* than $n=1$ on
  test (despite fitting train better) is a completely plausible, reportable
  finding, not a sign of a bug.
- **Do LSTM/GRU beat the averaged-vector models?** They have access to word
  *order*, which averaging discards — but language identification is a task
  where a handful of highly distinctive words (or even n-gram/character
  patterns) often carry most of the signal, so order may matter less here
  than in tasks like sentiment or syntax. Whether the extra architecture
  earns its keep on *this* dataset is an empirical question your results
  table already answers — report it plainly either way.
- **FastText supervised vs. everything else** is a genuinely different kind
  of model: it also uses **subword (character n-gram) features** natively
  (via `wordNgrams`/its internal handling), not just whole-word vectors, which
  is specifically why fastText tends to do well at language identification —
  a language's identity is often visible in short character sequences
  (e.g. "-tion" vs. "-ción" vs. "-tion") even in single unfamiliar words.
- **Macro-F1 vs. accuracy:** if your dataset has an uneven number of examples
  per language, check whether any classifier's ranking would change under
  plain accuracy instead of macro-F1 — a big gap between the two usually means
  the model is doing well on a majority language and poorly on a minority
  one, which macro-F1 is specifically designed to expose rather than hide.
